In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mdx2.io import loadobj
from mdx2.report.map_statistics import prepare_table

# load the site config
with open("site_config.yaml") as f:
    config = yaml.safe_load(f)

# project directory
project_directory = Path(config['project_directory'])

In [ ]:
# load data

DATASETS = {
    'CHESS 2025-11-05':'chess_20251105',
    'CHESS 2025-09-24':'chess_20250924',
    'ALS 2025-10-15':'als_20251015',
    'ALS 2025-11-25':'als_20251125',
}

data = {}

# load all the merged data
for dset_key, dset_root in DATASETS.items():
    merged_nxs = project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs'
    crystal = loadobj(merged_nxs, 'crystal')
    symmetry =  loadobj(merged_nxs, 'symmetry')
    hkl_table =  loadobj(merged_nxs, 'hkl_table')
    data[dset_key] = prepare_table(hkl_table, crystal, symmetry)

In [ ]:
# calculate stats binned by resolution

SMAX = 0.8

def calc_bin_edges(bin_width):
    nbins = int(SMAX/bin_width)
    return np.linspace(0, nbins*bin_width, nbins + 1)

def calc_correlations(df, key1, key2, bin_width=0.01):
    df = df.dropna(subset=[key1, key2])
    # compute statistics in each bin
    bin_edges = calc_bin_edges(bin_width)
    s_bins = pd.cut(df["s"], bins=bin_edges)
    df_isoavg = df.groupby(s_bins, observed=False).agg(
        s=("s", "mean"),
        cc=(key1, lambda x: x.corr(df.loc[x.index, key2])),
    )
    return df_isoavg

def cc_to_ccstar(cc):
    cc_star = np.sqrt(2*cc/(cc+1))
    return cc_star

correlations = {}

for key, df in data.items():
    print('Calculating cc 1/2 and cc* for dataset:',key)
    cc_all = calc_correlations(df, 'group_0_intensity_sub', 'group_1_intensity_sub')
    cc_excluded = calc_correlations(df[~df['is_reflection']], 'group_0_intensity_sub', 'group_1_intensity_sub')
    cc = cc_all.join(cc_excluded[['cc']], how='left', rsuffix='_excl')
    cc['cc_star'] = cc_to_ccstar(cc['cc'])
    cc['cc_excl_star'] = cc_to_ccstar(cc['cc_excl'])
    correlations[(key, key)] = cc

for key1, df1 in data.items():
    for key2, df2 in data.items():
        if key1>=key2:
            continue
        print('Calculating cc and cc limit between datasets:',key1,'and',key2)
        df12 = pd.merge(df1, df2, on=('h','k','l'), how='inner', suffixes=(None, '_2'))
        cc_all = calc_correlations(df12, 'intensity_sub', 'intensity_sub_2')
        cc_excluded = calc_correlations(df12[~df12['is_reflection']], 'intensity_sub', 'intensity_sub_2')
        cc = cc_all.join(cc_excluded[['cc']], how='left', rsuffix='_excl')
        cc1 = correlations[(key1,key1)]
        cc2 = correlations[(key2,key2)]
        cc['cc_limit'] = cc1['cc_star']*cc2['cc_star']
        cc['cc_excl_limit'] = cc1['cc_excl_star']*cc2['cc_excl_star']
        correlations[(key1, key2)] = cc

In [ ]:
def add_row_header(ax, row_header, row_pad=3, rotate_row_headers=True, **text_kwargs):
    ax.annotate(
        row_header,
        xy=(0, 0.5),
        xytext=(-ax.yaxis.labelpad - row_pad, 0),
        xycoords=ax.yaxis.label,
        textcoords="offset points",
        ha="right",
        va="center",
        rotation=rotate_row_headers * 90,
        **text_kwargs,
    )

def add_col_header(ax, col_header, col_pad=7, **text_kwargs):
    ax.annotate(
        col_header,
        xy=(0.5, 1),
        xytext=(0, col_pad),
        xycoords="axes fraction",
        textcoords="offset points",
        ha="center",
        va="baseline",
        **text_kwargs,
    )

In [ ]:
# Version using data with Bragg excluded (_excl)

fig, axs = plt.subplots(len(DATASETS), len(DATASETS), sharey=True, sharex=True, figsize=(10,10))

# sort datasets by date
ordered_keys = sorted(DATASETS.keys(), key=lambda x: x.split()[-1])

for i, key1 in enumerate(ordered_keys):
    for j, key2 in enumerate(ordered_keys):
        df = correlations.get((key1, key2), correlations.get((key2, key1), None))
        if df is not None:
            if key1==key2:
                df[['s','cc_excl_star']].set_index('s').plot(ax=axs[i][j],
                    legend=False,
                    color=['tab:purple'],
                    style=['-'],
                )
            elif j > i: # above the diagonal
                df[['s','cc_excl','cc_excl_limit']].set_index('s').plot(ax=axs[i][j],
                    legend=False,
                    color=['tab:blue','tab:cyan'],
                    style=['-','-'],
                )
                axs[i][j].fill_between(df["s"], df["cc_excl"], df["cc_excl_limit"], color="tab:cyan", alpha=0.2)
            else: # below the diagonal, skip it
                axs[i][j].axis('off')
                continue

for j, key in enumerate(ordered_keys):
    axs[j][j].set_xlim([0,SMAX])
    axs[j][j].set_ylim([0,1.05])
    axs[j][j].set_xlabel('1/d (Å$^{-1}$)')
    axs[j][j].set_ylabel('Correlation coefficient')
    axs[j][j].tick_params(labelbottom=True, labelleft=True)
    axs[j][j].xaxis.label.set_visible(True)
    axs[j][j].yaxis.label.set_visible(True)
    add_row_header(axs[j][j], key, fontsize="large")
    add_col_header(axs[0][j], key, fontsize="large")

fig.suptitle('Anisotropic component (Bragg voxels excluded)',fontsize=16)
plt.tight_layout()
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.show()

In [ ]:
# Dump the curves into csv files for reuse in manuscript figure
for (k1, k2), c in correlations.items():
    if k1==k2:
        df = c[['s','cc_excl_star']].rename(columns={"cc_excl_star": "cc_star"})
        fn = Path('figures/source_data')/f"fig2_cc_star_{DATASETS[k1]}.csv"
    else:
        df = c[['s','cc_excl','cc_excl_limit']].rename(columns={"cc_excl": "cc", "cc_excl_limit": "cc_limit"})
        fn = Path('figures/source_data')/f"fig2_cc_{DATASETS[k1]}_vs_{DATASETS[k2]}.csv"
    df = df[(df['s']<=0.81) & (df['s']>0.03)] # trim resolution range
    df = df.set_index('s')
    print('Writing',fn)
    df.to_csv(fn)

In [ ]:
correlations

In [ ]:
# Version using data with all voxels

fig, axs = plt.subplots(len(DATASETS), len(DATASETS), sharey=True, sharex=True, figsize=(10,10))

# sort datasets by date
ordered_keys = sorted(DATASETS.keys(), key=lambda x: x.split()[-1])

for i, key1 in enumerate(ordered_keys):
    for j, key2 in enumerate(ordered_keys):
        df = correlations.get((key1, key2), correlations.get((key2, key1), None))
        if df is not None:
            if key1==key2:
                df[['s','cc_star']].set_index('s').plot(ax=axs[i][j],
                    legend=False,
                    color=['tab:purple'],
                    style=['-'],
                )
            elif j > i: # above the diagonal
                df[['s','cc','cc_limit']].set_index('s').plot(ax=axs[i][j],
                    legend=False,
                    color=['tab:blue','tab:cyan'],
                    style=['-','-'],
                )
                axs[i][j].fill_between(df["s"], df["cc"], df["cc_limit"], color="tab:cyan", alpha=0.2)
            else: # below the diagonal, skip it
                axs[i][j].axis('off')
                continue

for j, key in enumerate(ordered_keys):
    axs[j][j].set_xlim([0,SMAX])
    axs[j][j].set_ylim([0,1.05])
    axs[j][j].set_xlabel('1/d (Å$^{-1}$)')
    axs[j][j].set_ylabel('Correlation coefficient')
    axs[j][j].tick_params(labelbottom=True, labelleft=True)
    axs[j][j].xaxis.label.set_visible(True)
    axs[j][j].yaxis.label.set_visible(True)
    add_row_header(axs[j][j], key, fontsize="large")
    add_col_header(axs[0][j], key, fontsize="large")

fig.suptitle('Anisotropic component (all voxels)',fontsize=16)
plt.tight_layout()
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.show()